# 02 — Iterative Pursuit

The intercept solution from Notebook 01 requires knowing the target's future position — which means trusting that the target will keep its heading and speed. 

What if you cannot predict that? Or simply do not want to?

**Pure pursuit** takes a different approach: at every moment, re-compute the bearing to the target's *current* position and steer directly toward it. No prediction. No math beyond what you already know. Just: look, aim, move, repeat.

It sounds reasonable. It even works — sometimes. But the path it produces is curved, the flight time is longer, and under certain conditions the pursuer never catches the target at all.

This notebook simulates both approaches side by side so you can see exactly what prediction buys you.

In [1]:
import math

def compute_bearing(p1, p2):
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def destination_point(origin, bearing_deg, distance_km):
    R = 6371.0
    d   = distance_km / R
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])
    lat2 = math.asin(math.sin(lat1)*math.cos(d) + math.cos(lat1)*math.sin(d)*math.cos(brg))
    lon2 = lon1 + math.atan2(math.sin(brg)*math.sin(d)*math.cos(lat1),
                              math.cos(d) - math.sin(lat1)*math.sin(lat2))
    return [math.degrees(lon2), math.degrees(lat2)]

# Module scenario
shooter_pos    = [-98.49, 33.91]
target_pos     = [-101.0, 36.5]
target_speed   = 300
target_heading = 135
shooter_speed  = 600

print("Ready.")

Ready.


## 1. The Pure Pursuit Loop

Pure pursuit is a simulation, not a formula. At each time step:

1. Compute the bearing from the pursuer's current position to the target's current position
2. Move the pursuer one step in that direction
3. Move the target one step along its heading
4. Check if the pursuer is within capture distance

Repeat until caught or the step limit is reached.

**Time step size** is a design choice with a real trade-off:
- Too large → the pursuer overshoots the target on close approach; path accuracy degrades
- Too small → more computation, but also more accurate curves

A good rule of thumb: the step distance (`speed × dt`) should be well under the capture radius. For our scenario, steps of `dt = 0.01 h` (36 seconds) move the pursuer ~6 km and the target ~3 km per step — small enough to be accurate.

In [2]:
def simulate_pursuit(shooter_pos, target_pos, target_heading, target_speed,
                     shooter_speed, dt=0.01, max_steps=2000, capture_km=5.0):
    """
    Simulate pure pursuit: pursuer always steers toward target's current position.

    Parameters
    ----------
    dt          : float, hours per time step
    max_steps   : int, step limit (prevents infinite loops)
    capture_km  : float, km — distance at which capture is declared

    Returns
    -------
    dict with:
        pursuer_path  : list of [lon, lat] — pursuer positions at each step
        target_path   : list of [lon, lat] — target positions at each step
        captured      : bool
        time_elapsed  : float, hours
        steps         : int
    """
    pursuer  = list(shooter_pos)
    target   = list(target_pos)
    p_path   = [list(pursuer)]
    t_path   = [list(target)]

    for step in range(max_steps):
        dist = haversine_km(pursuer, target)
        if dist <= capture_km:
            return {
                "pursuer_path": p_path,
                "target_path":  t_path,
                "captured":     True,
                "time_elapsed": step * dt,
                "steps":        step,
            }

        # Pursuer steers directly at current target position
        bearing = compute_bearing(pursuer, target)
        pursuer = destination_point(pursuer, bearing, shooter_speed * dt)

        # Target advances along its fixed heading
        target  = destination_point(target, target_heading, target_speed * dt)

        p_path.append(list(pursuer))
        t_path.append(list(target))

    return {
        "pursuer_path": p_path,
        "target_path":  t_path,
        "captured":     False,
        "time_elapsed": max_steps * dt,
        "steps":        max_steps,
    }


# Run the simulation
sim = simulate_pursuit(
    shooter_pos, target_pos, target_heading, target_speed, shooter_speed
)

status = "CAPTURED" if sim["captured"] else "NOT CAPTURED"
print(f"Result:         {status}")
print(f"Steps run:      {sim['steps']}")
print(f"Time elapsed:   {sim['time_elapsed']:.3f} h  ({sim['time_elapsed']*60:.1f} min)")
print(f"Path length:    {len(sim['pursuer_path'])} positions recorded")

Result:         CAPTURED
Steps run:      41
Time elapsed:   0.410 h  (24.6 min)
Path length:    42 positions recorded


## 2. Visualizing the Pursuit Curve

The pursuit path is not a straight line. Plot it alongside the target's path and the straight-line intercept solution from Notebook 01 and the shape becomes clear: the pursuer chases the target's tail, tracing a curve that converges from behind.

In [3]:
from ipyleaflet import Map, GeoJSON

def fc(features): return {"type": "FeatureCollection", "features": features}
def line(coords, props={}): return {"type":"Feature","geometry":{"type":"LineString","coordinates":coords},"properties":props}
def point(coord, props={}): return {"type":"Feature","geometry":{"type":"Point","coordinates":coord},"properties":props}

# Intercept solution (straight line) from Notebook 01
def find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd, t_max=10.0, tol=1e-6):
    def f(t):
        future = destination_point(t_pos, t_hdg, t_spd * t)
        return haversine_km(s_pos, future) - s_spd * t
    if f(t_max) > 0: return None
    lo, hi = 0.0, t_max
    for _ in range(60):
        mid = (lo + hi) / 2
        (lo if f(mid) > 0 else hi).__class__   # dummy — just use assignment below
        if f(mid) > 0: lo = mid
        else: hi = mid
        if hi - lo < tol: break
    return (lo + hi) / 2

t_int = find_intercept_time(shooter_pos, target_pos, target_heading, target_speed, shooter_speed)
intercept_pt = destination_point(target_pos, target_heading, target_speed * t_int)

# Subsample pursuit path for map performance
step = max(1, len(sim["pursuer_path"]) // 200)
pursuer_coords = sim["pursuer_path"][::step]
target_coords  = sim["target_path"][::step]

m = Map(center=(35.5, -99.5), zoom=6)

# Target path
m.add(GeoJSON(data=fc([line(target_coords)]),
              style={"color": "#457b9d", "weight": 1.5, "dashArray": "5"}))
# Pursuit curve (red)
m.add(GeoJSON(data=fc([line(pursuer_coords)]),
              style={"color": "#e63946", "weight": 2}))
# Intercept straight line (green)
m.add(GeoJSON(data=fc([line([shooter_pos, intercept_pt])]),
              style={"color": "#2a9d8f", "weight": 2, "dashArray": "6"}))

# Key points
m.add(GeoJSON(data=fc([point(shooter_pos)]),
              point_style={"radius": 7, "color": "#e63946", "fillOpacity": 1.0}))
m.add(GeoJSON(data=fc([point(target_pos)]),
              point_style={"radius": 7, "color": "#457b9d", "fillOpacity": 1.0}))
m.add(GeoJSON(data=fc([point(intercept_pt)]),
              point_style={"radius": 6, "color": "#2a9d8f", "fillOpacity": 1.0}))

print("Red curve  = pure pursuit path")
print("Green line = constant-velocity intercept (straight)")
print("Blue dashes = target path")
m

Red curve  = pure pursuit path
Green line = constant-velocity intercept (straight)
Blue dashes = target path


Map(center=[35.5, -99.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

The red curve bends toward the target and converges from behind. The green line goes straight to the intercept point. Both result in a catch — but the curved pursuit path is longer and takes more time.

## 3. Measuring the Cost of Pure Pursuit

How much extra distance and time does pure pursuit require compared to the straight intercept? The answer depends on the geometry: a target heading toward the pursuer costs little; a target heading away costs a lot.

In [4]:
def path_length_km(path):
    """Total distance along a list of [lon, lat] positions."""
    return sum(haversine_km(path[i], path[i+1]) for i in range(len(path) - 1))


# Compare pursuit vs intercept across target headings
intercept_tof    = t_int * 60                         # minutes
intercept_dist   = haversine_km(shooter_pos, intercept_pt)

pursuit_tof      = sim["time_elapsed"] * 60
pursuit_dist     = path_length_km(sim["pursuer_path"])

print("=== Baseline scenario (target heading 135°, incoming SE) ===")
print(f"{'':30} {'Intercept':>12}  {'Pursuit':>10}  {'Δ':>8}")
print("-" * 66)
print(f"{'Time of flight (min)':<30} {intercept_tof:>11.1f}   {pursuit_tof:>9.1f}  {pursuit_tof - intercept_tof:>+7.1f}")
print(f"{'Distance flown (km)':<30} {intercept_dist:>11.1f}   {pursuit_dist:>9.1f}  {pursuit_dist - intercept_dist:>+7.1f}")
print(f"{'Captured':<30} {'yes':>12}   {'yes' if sim['captured'] else 'NO':>9}")
print()

# Sweep target headings to show when pursuit fails
print("Pure pursuit success vs. target heading (shooter 600, target 300 km/h):")
print(f"  {'Heading':>9}  {'Captured':>10}  {'TOF (min)':>12}  {'Path (km)':>12}")
print("  " + "-" * 48)

for hdg in range(0, 360, 30):
    s = simulate_pursuit(shooter_pos, target_pos, hdg, target_speed, shooter_speed,
                         dt=0.01, max_steps=3000)
    tof_str  = f"{s['time_elapsed']*60:.1f}" if s["captured"] else "—"
    dist_str = f"{path_length_km(s['pursuer_path']):.1f}" if s["captured"] else "—"
    cap_str  = "yes" if s["captured"] else "NO"
    print(f"  {hdg:>8}°   {cap_str:>9}   {tof_str:>11}   {dist_str:>11}")

=== Baseline scenario (target heading 135°, incoming SE) ===
                                  Intercept     Pursuit         Δ
------------------------------------------------------------------
Time of flight (min)                  24.6        24.6     +0.0
Distance flown (km)                  245.5       246.0     +0.5
Captured                                yes         yes

Pure pursuit success vs. target heading (shooter 600, target 300 km/h):
    Heading    Captured     TOF (min)     Path (km)
  ------------------------------------------------
         0°         yes          67.2         672.0
        30°         yes          57.0         570.0
        60°         yes          45.0         450.0
        90°         yes          33.6         336.0
       120°         yes          25.8         258.0
       150°         yes          24.6         246.0
       180°         yes          29.4         294.0
       210°         yes          40.2         402.0
       240°         yes       

## 4. The Effect of Step Size

Step size controls the accuracy of the simulation. A large `dt` means the pursuer takes big leaps and the bearing is only recomputed infrequently — the curve becomes jagged and the capture time inflates. A small `dt` gives a smoother, more accurate path but takes more steps to compute.

The table below shows how the simulated capture time changes with `dt` for the same scenario.

In [5]:
dt_values = [0.1, 0.05, 0.02, 0.01, 0.005, 0.001]

print(f"{'dt (h)':>10}  {'Step dist (km)':>16}  {'TOF (min)':>12}  {'Steps':>8}  {'Captured':>10}")
print("-" * 64)
for dt in dt_values:
    max_s = int(5 / dt)   # cap at 5 simulated hours
    s = simulate_pursuit(shooter_pos, target_pos, target_heading, target_speed,
                         shooter_speed, dt=dt, max_steps=max_s)
    step_dist = shooter_speed * dt
    tof_str = f"{s['time_elapsed']*60:.2f}" if s["captured"] else "—"
    cap_str = "yes" if s["captured"] else "NO"
    print(f"{dt:>10.3f}   {step_dist:>14.1f} km  {tof_str:>11}   {s['steps']:>7}   {cap_str:>9}")

    dt (h)    Step dist (km)     TOF (min)     Steps    Captured
----------------------------------------------------------------
     0.100             60.0 km            —        50          NO
     0.050             30.0 km        33.00        11         yes
     0.020             12.0 km        26.40        22         yes
     0.010              6.0 km        24.60        41         yes
     0.005              3.0 km        24.30        81         yes
     0.001              0.6 km        24.30       405         yes


The capture time converges as `dt` decreases — that convergence is a sign the simulation is numerically stable. The `dt=0.01` value used throughout this notebook is a reasonable balance: accurate enough that the result is meaningful, fast enough to run interactively.

---

## Exercise A — Failure Modes

Set `shooter_speed = 250` and `target_speed = 300` (target is faster). Run `simulate_pursuit` for target headings `0°`, `90°`, `135°`, `180°`, and `270°`. 

For which headings does pure pursuit fail? For which does it still succeed? Explain in plain English why a slower pursuer can still catch a faster target under certain geometric conditions.

In [ ]:
test_headings = [0, 90, 135, 180, 270]

print(f"  {'Heading':>9}  {'Captured':>10}  {'TOF (min)':>12}")
print("  " + "-" * 36)
for hdg in test_headings:
    s = simulate_pursuit(shooter_pos, target_pos, hdg,
                         target_speed=300, shooter_speed=250,
                         dt=0.01, max_steps=5000)
    tof_str = f"{s['time_elapsed']*60:.1f}" if s["captured"] else "—"
    print(f"  {hdg:>8}°   {'yes' if s['captured'] else 'NO':>9}   {tof_str:>11}")

# your explanation here (as a comment)
# This code tests the pursuit simulation for a set of target headings and prints the results.
# 
# 1. `test_headings`: A list of target headings (in degrees) to test, representing the direction the target moves.
#    - The headings are 0° (north), 90° (east), 135° (southeast), 180° (south), and 270° (west).
#
# 2. The `print` statements format and display a table header with columns for:
#    - "Heading": The target's heading in degrees.
#    - "Captured": Whether the shooter successfully intercepted the target ("yes" or "NO").
#    - "TOF (min)": The time of flight (TOF) in minutes, which is the time it took to capture the target.
#      If the target was not captured, this column displays "—".
#
# 3. The `for` loop iterates over each heading in `test_headings`:
#    - `simulate_pursuit`: A function that simulates the pursuit scenario.
#      - `shooter_pos` and `target_pos`: The initial positions of the shooter and target.
#      - `hdg`: The current heading of the target.
#      - `target_speed` and `shooter_speed`: The speeds of the target and shooter (in consistent units, e.g., km/h).
#      - `dt`: The time step for the simulation (0.01 units per step).
#      - `max_steps`: The maximum number of simulation steps to prevent infinite loops.
#    - The function returns a dictionary `s` with:
#      - `s["captured"]`: A boolean indicating whether the shooter captured the target.
#      - `s["time_elapsed"]`: The time elapsed during the simulation (in consistent units, e.g., hours).
#
# 4. `tof_str`: A formatted string for the time of flight (TOF) in minutes.
#    - If the target was captured, the time elapsed is converted to minutes (`s["time_elapsed"]*60`) and formatted to 1 decimal place.
#    - If the target was not captured, "—" is displayed.
#
# 5. The `print` statement outputs the results for each heading in a formatted table row:
#    - The heading (e.g., "90°").
#    - Whether the target was captured ("yes" or "NO").
#    - The time of flight (TOF) in minutes or "—" if not captured.

## Exercise B — Map Multiple Pursuit Curves

Run `simulate_pursuit` for four different target headings (`45°`, `135°`, `225°`, `315°`) against the same shooter. Plot all four pursuit paths on a single ipyleaflet map using a different color per heading. Add the target start position and the shooter position as markers.

Which curves are shortest? Which are longest? Does the shape of the curve tell you anything about the geometry?

In [12]:
headings_to_plot = [45, 135, 225, 315]
colors = ["#e63946", "#2a9d8f", "#e9c46a", "#457b9d"]

# your code here
# hint: subsample each path (every Nth point) before adding to the map

## Exercise C — Pursuit vs. Intercept Time Budget

For the baseline scenario, the intercept solution takes less time than pure pursuit. How much less depends on target heading. 

For each heading in `range(0, 360, 15)`:
1. Run `simulate_pursuit` and record the pursuit time (or `None` if it fails)
2. Run `find_intercept_time` and record the intercept time (or `None`)
3. Compute the time saved by using intercept over pursuit

Print a table and identify: which heading maximizes the time savings? Which heading makes pursuit and intercept nearly equivalent?

In [13]:
# your code here
# hint: use shooter_speed=600, target_speed=300 for both methods
shooter_pos   = [-98.49, 33.91]
target_pos    = [-97.0,  30.0]
target_speed  = 300
shooter_speed = 600

print(f"{'Heading':>8} {'Pursuit(h)':>12} {'Intercept(h)':>14} {'Saved(h)':>10} {'Saved(min)':>11}")
print("-" * 60)

best_savings  = -1
best_heading  = None
equiv_heading = None
equiv_diff    = float("inf")

for heading in range(0, 360, 15):

    # Pursuit time — get last step time from simulate_pursuit
    try:
        path = simulate_pursuit(shooter_pos, target_pos, heading, target_speed, shooter_speed)
        pursuit_t = path[-1][2] if path else None  # adjust index if needed
    except:
        pursuit_t = None

    # Intercept time
    try:
        intercept_t, _ = verified_intercept(
            shooter_pos, target_pos, heading, target_speed, shooter_speed
        )
    except:
        intercept_t = None

    # Time saved
    if pursuit_t and intercept_t:
        saved = pursuit_t - intercept_t
        saved_min = saved * 60

        if saved > best_savings:
            best_savings = saved
            best_heading = heading

        diff = abs(saved)
        if diff < equiv_diff:
            equiv_diff    = diff
            equiv_heading = heading

        print(f"{heading:>8}°  {pursuit_t:>11.3f}  {intercept_t:>13.3f}  {saved:>9.3f}  {saved_min:>10.1f}")
    else:
        p_str = f"{pursuit_t:.3f}" if pursuit_t else "None"
        i_str = f"{intercept_t:.3f}" if intercept_t else "None"
        print(f"{heading:>8}°  {p_str:>11}  {i_str:>13}  {'N/A':>9}  {'N/A':>10}")

print()
print(f"Max time savings:          heading={best_heading}°  ({best_savings*60:.1f} min saved)")
print(f"Pursuit ≈ Intercept:       heading={equiv_heading}°  (diff={equiv_diff*60:.1f} min)")

 Heading   Pursuit(h)   Intercept(h)   Saved(h)  Saved(min)
------------------------------------------------------------
       0°         None           None        N/A         N/A
      15°         None           None        N/A         N/A
      30°         None           None        N/A         N/A
      45°         None           None        N/A         N/A
      60°         None           None        N/A         N/A
      75°         None           None        N/A         N/A
      90°         None           None        N/A         N/A
     105°         None           None        N/A         N/A
     120°         None           None        N/A         N/A
     135°         None           None        N/A         N/A
     150°         None           None        N/A         N/A
     165°         None           None        N/A         N/A
     180°         None           None        N/A         N/A
     195°         None           None        N/A         N/A
     210°         None   

---

## Check Your Understanding

A student runs `simulate_pursuit` with `dt=0.1` and gets a capture time of 28 minutes. They run it again with `dt=0.001` and get 24 minutes. They conclude the simulation is broken because it gives different answers.

**Question:** Is the simulation broken? Explain why the two results differ, which one is more accurate, and how you would determine when `dt` is small enough that further reduction stops mattering.

```python
# your answer here
```
the simulation isn't broken, it's converging

---

## Next

In [03 — Visual Simulation](./03-Visual_Simulation.ipynb), we bring both strategies to life — click to place the shooter and target, set the motion, and watch intercept and pursuit animate side by side on a live map.